## IGA Preprocessing Script for MongoDB

Author: Anna Margarita Sofia T. Licup (Margie)

Last Modified: 09/05/2026

Note: Rebekah De Losa's MongoDBUploadScript_Woolworths.ipynb was the base for this. Modifications were made to fit the IGA scraped file.

---

### Purpose
This script takes raw IGA product pricing data (scraped as CSV files), cleans and standardizes it, matches each product to the DiscountMate product database, and uploads the pricing records to MongoDB.

**Run this script whenever new IGA scrape files are available and need to be loaded into the MongoDB database.**

### Steps
1. **Setup** - import libraries, set the data folder path, connect to MongoDB
2. **Define helper functions** - database read/write custom functions
3. **Define target columns** - the fields required by the database schema
4. **Define processing functions** - clean and reformat each column
5. **Load & process files** - loop through all CSVs, apply processing functions, compile the results
6. **Validate** - confirm that the data is clean and complete before upload
7. **Deduplicate against MongoDB** - remove any records already in the database to avoid re-uploading
8. **Upload to MongoDB** - send the processed records to the `product_pricings` collection in MongoDB

### Before you run this script
- Make sure you have a `.env` file in the **same folder as this notebook** containing your MongoDB connection string.
- Update the `path` variable in **Step 1b** to point to your local folder containing the scraped CSV files.
- Required Python packages: `pandas`, `numpy`, `pymongo`, `python-dotenv`

### Step 1a - Setup: Import Libraries
We import all the libraries needed before running anything else. We need to import these to make sure that we're able to use the commands from the library. 

In [34]:
# Libraries for preprocessing the data

# Imports pandas library to work with the data like a spreadsheet where there are rows and columns
import pandas as pd

# Imports the EmptyDataError to skip over any files that are empty (no to crashing!)
from pandas.errors import EmptyDataError

# Imports the numpy library to handle numbers and fill in blanks where data is missing
import numpy as np

# Imports the os library to find and open files from folders on the computer
import os

# Imports datetime and timezone to help when recording exactly when the data was uploaded
from datetime import datetime, timezone

# Imports MongoClient to connect to the MongoDB database
from pymongo import MongoClient

# Imports load_dotenv to load my database password from a hidden file (for security!)
from dotenv import load_dotenv

### Step 1b - Set-Up: Set Data Folder Path

**Update this path before running.** Point it to the folder on your machine that contains the scraped store subfolders (ex. `iga/`, `coles/`).

This creates a variable so we don't have to keep manually updating the path in all succeeding code blocks.


In [35]:
# IGA Path based on local storage
# Creates a variable so we don't have to keep manually updating the path in all succeeding code blocks
path = r"C:\Users\margi\Capstone T1 2026\DiscountMate_new\experimental\WebAPIScraping"

### Step 1c - Set Up: Connect to MongoDB

We load the database password from the `.env` file (so it is never written directly into this script) and use it to connect to the DiscountMate MongoDB database.

FYI: If the connection fails, check that your `.env` file exists in the same folder as this notebook and contains a valid `MONGO_URI` value.

In [36]:
# Loads the .env file so that we can easily use the database password
load_dotenv()

# Gets the database password from the .env file and stores it in a variable for easy access
mongo_uri = os.getenv('MONGO_URI')

# Uses the password to connect to the MongoDB database, like logging in
client = MongoClient(mongo_uri)

# Decides which database we want to use for this
database = 'DiscountMate_DB'

#  Opens the database
db = client[database]

### Step 2 - Define Helper Functions

These two functions handle reading from and writing to MongoDB:

1.  **get_data** - retrieves all records from a given collection and returns them as a pandas DataFrame
2.  **create_data** - uploads a list of processed records to a given collection one by one, printing progress so you can monitor the upload and see where it stopped if it fails midway

> **Note:** update_data is reserved for future use (editing existing records) and is not yet implemented.

In [37]:
# Defines functions to use when retrieving the scraped data and uploading the files
def get_data(table, database):
    # Pulls all records from a given table in the database and turns it into a spreadsheet
    return pd.DataFrame(database[table].find({}))

# Uploads the processed data to the MongoDB database (and the progress of upload)
def create_data(db, tablename, newDocuments):
    # newDocuments: a list of processed records ready to be uploaded to the database
    # Starts a counter to keep track of how many records have been uploaded
    docCount = 0                    
    # Opens the target table in the database    
    collection = db[tablename]    
    # Goes through each record one by one      
    for doc in newDocuments:    
        # Uploads the current record to the database   
        collection.insert_one(doc)    
        # Adds 1 to the counter after each successful upload   
        docCount += 1
        # Prints the current count so we can see the upload progress (out of how many are being uploaded and so we know where it fails if ever)
        print(f"{docCount}/{len(newDocuments)} uploaded - product_id: {doc['product_id']}, date: {doc['date']}")
    print(f"Upload complete. {docCount} records uploaded.")
    # Reloads the table to confirm everything was uploaded      
    table = pd.DataFrame(collection.find({}))
    # Returns the updated table and the total number of records uploaded
    return table, docCount

# Placeholder for a tool for edits maybe??? (Leaving this here)
def update_data():                            
    return

# Load the existing products table from the database so we can match IGA products to it later
prods = get_data('products', db)

### Step 3 - Define Target Columns
This is the list of columns that the product_pricings collection in MongoDB expects. Every record we upload must contain these fields.

The processing functions in Step 4 are responsible for producing each of these columns from the raw IGA data.

In [38]:
# Defines the list of columns we want to keep and upload to the database
# Checklist of info for the database
targetvars = ['product_id',      # The unique ID that identifies the product in our database
              'product_code',    # Code for the product
              'store_chain',     # The store the product was scraped from
              'date',            # The date the price was collected from the IGA website
              'price',           # The price of the product on that date
              'best_price',      # The lowest price ever seen for this product (historically)
              'best_unit_price', # The lowest unit price ever seen for this product (historically)
              'unit_price',      # The price per unit on that date
              'is_on_special',   # Whether the product was on special on that date (True or False)
              'created_at']      # The date and time this record was uploaded to the database

### Step 4 - Define Processing Functions

These functions clean and reformat the raw IGA data into the standard format used by the database:

1. **get_date** - converts the scrape timestamp into a consistent date and time format
2. **get_price** - ensures the product price is stored as a number and not text
3. **get_unit_price** - calculates the price per single unit (ex. '0.0147 per g') by dividing the unit price by its quantity. If the unit price column is blank, it calculates the price using the product price and size instead. If the unit label is also missing, it gets it from the unit type column.
4. **transform_barcode** - cleans the barcode so it can be matched to products in the database
5. **get_is_on_special** - flags whether the product was on a temporary price reduction on that date
6. **join_prods** - matches each IGA record to an existing product in the database using the barcode

In [39]:
def get_date(data, coltarget):
    # Reads the date column so Python can understand and use it as a date
    # Margie Function"""
    # Converts the date column into the correct format so Python can read it
    # data['date'] = pd.to_datetime(data[coltarget], format='%Y-%m-%d %H:%M:%S')
    # # Reformats the date and time to match the database (ex. '29/03/2026 23:21')
    # data['date'] = data['date'].dt.strftime('%d/%m/%Y %H:%M')
    #Bek function
    #Explanation: Data needs to be in timestamp format. Updated this in my script as well
    data['date'] = pd.to_datetime(data[coltarget])

def get_price(data, coltarget):
    # Makes sure the price is saved as a number with decimals (not as text)
    data['price'] = data[coltarget].astype('float')

def get_unit_price(data, unit_price_col, size_col, unit_col):
    # Standardizes the unit names (ex. 'grams' to g)
    size_dict = {
        'gram': 'g',
        'grams': 'g',
        'litre': 'l',
        'litres': 'l',
        'metre': 'm',
        'metres': 'm',
        'meters': 'm',
        'millilitre': 'ml',
        'each': 'each',
    }
    # Pulls out the price, quantity, and unit from the unit price column (ex '$1.47/100g' so that it becomes '1.47', '100', 'g')
    extracted = data[unit_price_col].str.extract(r'\$([\d\.]+)/([\d\.]+)\s*(\w+)')
    price_value = extracted[0].astype(float)
    unit_quantity = extracted[1].astype(float)
    unit_label = extracted[2].str.lower().str.strip().replace(size_dict)
    # Gets the price for one single unit by dividing (ex. $1.47 for 100g = $0.0147 per g)
    data['unit_value'] = round(price_value / unit_quantity, 3)
    data['unit_label'] = unit_label
    # If the unit price column is empty, determine the product price and size instead
    data['unit_value'] = np.where(
        data['unit_value'].isna(),
        round(data['price'] / data[size_col].astype(float), 3),
        data['unit_value']
    )
    # If the unit label is still missing, use the unit column as a backup file
    data['unit_label'] = np.where(
        data['unit_label'].isna(),
        data[unit_col].str.lower().replace(size_dict),
        data['unit_label']
    )
    # Joins the price and unit together into a readable format (ex. '0.0147 per g', '2.50 per each')
    data['unit_price'] = data['unit_value'].astype(str) + ' per ' + data['unit_label']
    # Removes the helper columns that were only needed for the calculation
    data.drop(columns=['unit_value', 'unit_label'], inplace=True)

def transform_barcode(data, coltarget):
    # If a product has more than one barcode listed, only keep the first one
    # Will bring this up with the tam
    data['gtin'] = data[coltarget].astype(str).str.split(',').str[0].str.strip()
    # Treats 'nan' and '0' as blank since they are not real barcodes
    data['gtin'] = data['gtin'].replace({'nan': None, '0': None})
    # Removes any products with no barcode since they cannot be matched to the database
    data.dropna(subset=['gtin'], inplace=True)

def get_is_on_special(data, coltarget):
    # Flags a product as on special (True) if IGA has marked it as a temporary price reduction ('tpr')
    # Everything else, including blank values, is marked as not on special (False)
    data['is_on_special'] = data[coltarget].str.lower() == 'tpr'

def join_prods(data, product_data):
    # Matches each product to its entry in our database using the barcode as the common link
    # Any products that don't have a match in the database are removed
    return pd.merge(product_data[['_id', 'product_code', 'gtin']].copy(),
                    data,
                    how='inner',
                    on='gtin').rename(columns={'_id': 'product_id'})

### Step 5 - Load and Process IGA Files
1. Defines a column rename mapping to handle older IGA files that used different column names (ex. scrapedat instead of scraped_at). Files with updated names are unaffected.
2. Loops through every CSV in the iga/ subfolder, skipping non-CSV or empty files.
3. Applies all processing functions to each file.
4. Matches each record to the products table using the barcode.
5. Concatenates all processed files into a single DataFrame ready for upload.

In [40]:
# Older IGA files used different column names to newer ones (ex. 'scrapedat' vs 'scraped_at').
# This table renames the old column names to match the new format since the new format will be the naming convention moving forward.
old_column_rename_mapping = {
    'scrapedat': 'scraped_at',
    'iga_pricenumeric': 'iga_price_numeric',
    'iga_priceperunit': 'iga_price_per_unit',
    'iga_pricelabel': 'iga_price_label',
    'iga_pricesource': 'iga_price_source',
    'iga_tprprice': 'iga_tpr_price',
    'iga_wholeprice': 'iga_whole_price',
    'iga_wasprice': 'iga_was_price',
    'iga_waspricenumeric': 'iga_was_price_numeric',
    'iga_waswholeprice': 'iga_was_whole_price',
    'iga_productid': 'iga_product_id',
    'iga_isfavorite': 'iga_is_favorite',
    'iga_ispastpurchased': 'iga_is_past_purchased',
    'iga_sellby': 'iga_sell_by',
    'iga_defaultcategory': 'iga_default_category',
    'iga_weightincrement': 'iga_weight_increment',
    'iga_shoppingrulemessages': 'iga_shopping_rule_messages',
    'iga_shoppingrulemessages_blocking': 'iga_shopping_rule_messages_blocking',
    'iga_shoppingrulemessages_information': 'iga_shopping_rule_messages_information',
    'iga_unitofmeasure_abbreviation': 'iga_unit_of_measure_abbreviation',
    'iga_unitofmeasure_label': 'iga_unit_of_measure_label',
    'iga_unitofmeasure_size': 'iga_unit_of_measure_size',
    'iga_unitofmeasure_type': 'iga_unit_of_measure_type',
    'iga_unitofprice_abbreviation': 'iga_unit_of_price_abbreviation',
    'iga_unitofprice_label': 'iga_unit_of_price_label',
    'iga_unitofprice_size': 'iga_unit_of_price_size',
    'iga_unitofprice_type': 'iga_unit_of_price_type',
    'iga_unitofsize_abbreviation': 'iga_unit_of_size_abbreviation',
    'iga_unitofsize_label': 'iga_unit_of_size_label',
    'iga_unitofsize_size': 'iga_unit_of_size_size',
    'iga_unitofsize_type': 'iga_unit_of_size_type',
    'iga_citrusad_adid': 'iga_citrus_ad_ad_id',
    'brandname': 'brand_name',
    'brandquery': 'brand_query',
    'storeid': 'store_id',
    'sellby': 'sell_by',
    'pricedisplay': 'price_display',
    'pricenumeric': 'price_numeric',
    'pricelabel': 'price_label',
    'pricesource': 'price_source',
    'priceperunit': 'price_per_unit',
    'primaryimageurl': 'primary_image_url',
    'waspricedisplay': 'was_price_display',
    'waspricenumeric': 'was_price_numeric',
    'runid': 'run_id',
    'rawjson': 'raw_json',
}

In [41]:
# Creates an empty container to store each CSV file when it's loaded
df_list = {}              
# The name of the store folder to look for CSV files in
store = 'iga'             

# Loops through every file in the IGA folder
for file in os.listdir(os.path.join(path, store)):   
    # Skips anything that isn't a CSV file
    if not file.endswith('.csv'):                     
        continue
    try:
        # Reads the CSV, keeping the barcode as text to avoid scientific notation
        df = pd.read_csv(os.path.join(path, store, file), low_memory=False, dtype={'iga_barcode': str})  
        # Adds the loaded file to the container
        df.columns = df.columns.str.strip().str.lower()
        if any(col in df.columns for col in old_column_rename_mapping.keys()):
            df.rename(columns=old_column_rename_mapping, inplace=True)
        df_list[file] = df                            
        print(f"File {file} added to dataframe list...")
    except EmptyDataError:
        # Skips any CSV files that are completely empty
        print(f"File {file} is empty. Skipping...")  

# # What each column in the raw file maps to in the database:
# #   scraped_at          -> date           (format: 'dd/mm/yyyy HH:MM') date and time the price was collected
# #   iga_price_numeric   -> price          (already a number, no conversion needed) current price of the product
# #   iga_price_per_unit   -> unit_price     (formatted string :'price per unit') price per unit 
# #   iga_barcode        -> gtin           (read as string to avoid scientific notation) barcode
# #   iga_pricesource    -> is_on_special  (True if the value is 'tpr', otherwise False) 'tpr' means it's on special

# Create empty df for concatenating transformed data into
complete_df = pd.DataFrame(columns=targetvars)

# Transform dataframes and concatenate into complete_df for upload
for name, df in df_list.items():
    # Prints how many rows are in the file
    print(f"Row count: {len(df)}")

    # Applies the functions for preprocessing
    get_date(df, 'scraped_at')
    get_price(df, 'iga_price_numeric')
    get_unit_price(df, 'iga_price_per_unit', 'iga_unit_of_measure_size', 'iga_unit_of_measure_type')
    transform_barcode(df, 'iga_barcode')
    get_is_on_special(df, 'iga_price_source')             

    # Matches each record to an existing product in the database
    joindf = join_prods(df, prods)        
    # Placeholder: need to talk to DA/ML team for definition                                                       
    joindf[['best_price', 'best_unit_price']] = None          
    # Labels all records as IGA so we know which store they came from                                   
    joindf['store_chain'] = 'iga_generic'      

    """Bek Edit: Get target vars"""
    # Restict to just the variables of interest
    joindf = joindf[targetvars[:-1] + ['gtin']].copy()                                                  

    """Bek Edit: Moved duplicate dropping to make sure it is on a file level"""
    print(f"{name} has initial upload length {len(joindf)}")
    # Removes any duplicate rows based on barcode (might be an issue if there are same barcode but different prices running at a time)
    joindf.drop_duplicates(subset=['gtin'], inplace=True)  
    print(f"{name} has upload length {len(joindf)} after dropping gtin duplicates")

    # Prints how many rows were successfully matched to the database after preprocessing
    print(f"{name} has upload length {len(joindf)}")

    # Concat with other transformed files
    complete_df = pd.concat([complete_df, joindf], ignore_index=True)

File iga_all_products_20260406_105542.csv added to dataframe list...
File iga_all_products_20260414_022942.csv added to dataframe list...
File iga_all_products_20260419_124225.csv added to dataframe list...
Row count: 20383
iga_all_products_20260406_105542.csv has initial upload length 7915
iga_all_products_20260406_105542.csv has upload length 7915 after dropping gtin duplicates
iga_all_products_20260406_105542.csv has upload length 7915
Row count: 20361


C:\Users\margi\AppData\Local\Temp\ipykernel_15176\1226781401.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  complete_df = pd.concat([complete_df, joindf], ignore_index=True)


iga_all_products_20260414_022942.csv has initial upload length 7896
iga_all_products_20260414_022942.csv has upload length 7896 after dropping gtin duplicates
iga_all_products_20260414_022942.csv has upload length 7896
Row count: 20474
iga_all_products_20260419_124225.csv has initial upload length 7882
iga_all_products_20260419_124225.csv has upload length 7882 after dropping gtin duplicates
iga_all_products_20260419_124225.csv has upload length 7882


### Step 6 - Validate Data Before Uploading

Run these checks before uploading to confirm the processed data is clean and in the correct format. If any check fails, investigate the issue before proceeding to Step 7.

1. **Target columns present** - confirms that all required columns exist in the dataframe
2. **No nulls in critical fields** - checks that product_id, product_code, store_chain, date, price and gtin are not blank
3. **Price is valid** - checks that no prices are zero or negative
4. **Date format is correct** - confirms the date is in the expected dd/mm/yyyy HH:MM format
5. **is_on_special is boolean** - confirms the is_on_special flag is stored as True or False
6. **store_chain is consistent** - confirms all records are labelled as 'iga_generic'

If there are any issues, it'll be printed out so it can be checked.

In [43]:
# Checks if all target columns are there
# Excludes created_at as this is added at upload
upload_cols = targetvars[:-1] + ['gtin'] 
# Finds any columns that are missing from the dataframe     
missing_cols = [c for c in upload_cols if c not in complete_df.columns]  
# Prints the missing columns, or confirms all are present
print(f"Missing target columns: {missing_cols if missing_cols else 'None'}")  

# Checks for null values
# The fields that must not be blank
necessary = ['product_id', 'product_code', 'store_chain', 'date', 'price', 'gtin']
# Counts how many blank values exist in each column
null_counter = complete_df[necessary].isnull().sum()          
print("\nNo. of null in required fields:")
# Prints the blank count for each necessary field
print(null_counter)                                     

# Checks if price is numeric and positive (no negatives!)
# Finds any records where the price is zero or negative
invalid_prices_iga = complete_df[complete_df['price'] <= 0]         
# Prints how many invalid prices were found 
print(f"\nRecords with price <= 0: {len(invalid_prices_iga)}")  

# Checks if date format is correct
try:
    # Tries to parse the date column using the needed format for the database
    pd.to_datetime(complete_df['date'], format='%d/%m/%Y %H:%M') 
    # Confirms the format is correct if there is no error message
    print("\nDate format check: SUCCESS")   
except Exception as e:
    # Prints the error if the format is incorrect
    print(f"\nDate format check: FAILED {e}")              

# Checks the is_on_special is in the correct format (Boolean: True or False)
# Prints the data type of the is_on_special column
print(f"\nis_on_special dtype: {complete_df['is_on_special'].dtype}")             
# Prints how many True and False values exist for that column
print(f"is_on_special counter:\n{complete_df['is_on_special'].value_counts()}")

# Checks if store_chain values are all iga_generic
# Prints the unique values to confirm they are all 'iga_generic'
print(f"\nstore_chain unique values: {complete_df['store_chain'].unique()}")      

# Checks the final shape of the processed file
# Prints the number of rows and columns in the final dataframe
print(f"\nFinal dataframe shape: {complete_df.shape}")  
# Shows the first 5 rows of the final dataframe
complete_df[upload_cols].head()                         

Missing target columns: None

No. of null in required fields:
product_id      0
product_code    0
store_chain     0
date            0
price           0
gtin            0
dtype: int64

Records with price <= 0: 0

Date format check: SUCCESS

is_on_special dtype: object
is_on_special counter:
is_on_special
False    15565
True      8128
Name: count, dtype: int64

store_chain unique values: ['iga_generic']

Final dataframe shape: (23693, 11)


,product_id,product_code,store_chain,date,price,best_price,best_unit_price,unit_price,is_on_special,gtin
0,696f64f96b7787e691e79020,pub style extra crispy fries_mccain_750g,iga_generic,2026-04-06 21:03:29,5.85,4.80,0.006 per g,0.008 per g,True,9310174173358
1,696f64fa6b7787e691e79022,snack chips mix lunchbox snacks multipack asso...,iga_generic,2026-04-06 21:09:45,8.00,8.00,8.0 per each,8.0 per each,True,9310015249037
2,696f64fa6b7787e691e79023,kids strawberry yoghurt pouch_vaalia_140g,iga_generic,2026-04-06 20:56:05,2.50,2.50,0.018 per g,0.018 per g,False,9310036041139
3,696f64fa6b7787e691e79024,fresh beef ravioli pasta 4 serve_latina_625g,iga_generic,2026-04-06 21:05:00,8.00,5.50,0.009 per g,0.013 per g,True,9312353000875
4,696f64fa6b7787e691e79025,chips natural_vege_100g,iga_generic,2026-04-06 21:15:33,5.95,5.95,0.06 per g,0.06 per g,False,9315991022001


### Step 8 - Deduplicate Against MongoDB (Skip Already-Uploaded Records)

It's common for scrape files to accumulate in the folder over time, meaning the same file may be present across multiple runs of this script. This step protects against uploading duplicate records by checking what is already in MongoDB and keeping only records that haven't been uploaded yet.

**Steps:**
1. We fetch the product_id and date of every existing IGA record from MongoDB (just these two fields and not the full records, so it's fast).
2. We then do a left join between our local processed data and the existing MongoDB records.
3. Any local record that already has a matching product_id + date combination in MongoDB is dropped. Only genuinely new records proceed to upload.

> A record is considered a duplicate if it has the same product_id **and** date as something already in the database. Two records for the same product on different dates are treated as distinct pricing snapshots and are both kept.

In [ ]:
# Fetch the product_id and date of all existing IGA records already in MongoDB.
# We only retrieve these two fields (not full records) to keep this fast.
existing_iga = pd.DataFrame(db['product_pricings'].find(
    {'store_chain': 'iga_generic'},
    {'product_id': 1, 'date': 1, '_id': 0}
))

if len(existing_iga) > 0:
    # Ensure both sides use the same datetime format before comparing
    existing_iga['date'] = pd.to_datetime(existing_iga['date'])
    complete_df['date'] = pd.to_datetime(complete_df['date'])

    # Left join — records that exist in MongoDB will get a '_merge' value of 'both'
    merged = complete_df.merge(existing_iga, on=['product_id', 'date'], how='left', indicator=True)

    already_uploaded = merged[merged['_merge'] == 'both']
    to_upload = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

    print(f"Records already in MongoDB (will be skipped): {len(already_uploaded)}")
    print(f"New records to upload: {len(to_upload)}")

    # Replace complete_df with only the new records
    complete_df = to_upload
else:
    # No existing IGA records found so this is likely the first run, upload everything
    print("No existing IGA records found in MongoDB — uploading all records.")
    print(f"Records to upload: {len(complete_df)}")

### Step 9 - Prepare and Upload to MongoDB

**Steps**
1. Copies the processed DataFrame and drops the gtin column (used for joining only as it's not stored in MongoDB)
2. Adds a created_at timestamp so we know when each record was uploaded
3. Replaces Na values with None (MongoDB does not accept Python's NaN)
4. Converts the DataFrame to a list of dictionaries for upload
5. Uploads all new records to the product_pricings collection

> If the upload fails partway through, **re-run from Step 8**: create_data inserts one record at a time and prints progress, so you can see exactly where it stopped. Step 8 will automatically exclude anything already uploaded, so there is no risk of double-uploading.

In [46]:
import_data = complete_df.copy()
# Remove unnecessary columns: gtin is just for joining, not required for upload
import_data.drop(columns=['gtin'], inplace=True, errors='ignore')
# Add created at timestamp
import_data['created_at'] = pd.to_datetime('now')
# Replace NaN with None for MongoDB
import_data = import_data.replace({np.nan: None})
# Convert to list of dictionaries for upload
import_dict = import_data.to_dict(orient='records')
print(f"Import dict has {len(import_dict)} records")

Import dict has 23693 records


In [47]:
# Uploads all records in import_dict to the product_pricings table in MongoDB one by one
updatedPriceTable, priceCount = create_data(db, 'product_pricings', import_dict)
# Prints the total number of records successfully uploaded
print(f"Upload finished. {priceCount} product_pricings uploaded.")

1/23693 uploaded - product_id: 696f64f96b7787e691e79020, date: 2026-04-06 21:03:29
2/23693 uploaded - product_id: 696f64fa6b7787e691e79022, date: 2026-04-06 21:09:45
3/23693 uploaded - product_id: 696f64fa6b7787e691e79023, date: 2026-04-06 20:56:05
4/23693 uploaded - product_id: 696f64fa6b7787e691e79024, date: 2026-04-06 21:05:00
5/23693 uploaded - product_id: 696f64fa6b7787e691e79025, date: 2026-04-06 21:15:33
6/23693 uploaded - product_id: 696f64fa6b7787e691e79026, date: 2026-04-06 21:16:04
7/23693 uploaded - product_id: 696f64fb6b7787e691e79027, date: 2026-04-06 20:56:05
8/23693 uploaded - product_id: 696f64fb6b7787e691e79028, date: 2026-04-06 21:16:04
9/23693 uploaded - product_id: 696f64fb6b7787e691e79029, date: 2026-04-06 21:01:25
10/23693 uploaded - product_id: 696f64fb6b7787e691e7902a, date: 2026-04-06 20:56:05
11/23693 uploaded - product_id: 696f64fb6b7787e691e7902b, date: 2026-04-06 21:13:54
12/23693 uploaded - product_id: 696f64fb6b7787e691e7902d, date: 2026-04-06 21:01:24
1

---

### Known Data Issues and How They Were Handled

Updated as of May 9 2026

This section documents the known issues found in the IGA scraped data and the fixes applied in this script. 

---

#### 1. Two different column naming conventions across files
**Issue:**  
IGA scrape files from the first 6 weeks used column names without underscores (e.g. `scrapedat`, `iga_pricenumeric`). Files from the last 2 weeks switched to snake_case (e.g. `scraped_at`, `iga_price_numeric`). Both formats exist in the folder at the same time.

**Fix:**  
At load time, all column names are lowercased and stripped of whitespace, then an `old_column_rename_mapping` dictionary renames any old-style columns to the new standard before processing begins. Files that already use the new naming convention are unaffected. See the rename mapping in Step 5.

---

#### 2. Some products have two barcodes in a single row
**Issue:**  
The `iga_barcode` column occasionally contains two barcodes in one cell, separated by a comma (ex. `"9310015513102, 9310015513119"`). The products table in MongoDB holds only one barcode per product, so a two-barcode value will fail to match.

**Temp Fix:**  
Only the first barcode is used for matching. This means if the first barcode doesn't exist in the products table but the second does, the record will be dropped.

**To investigate:**  
Whether the second barcode should be tried as a fallback if the first doesn't match. Raise with the team before changing this behaviour.

---

#### 3. `iga_price_per_unit` column is sometimes blank
**Issue:** 
The `iga_price_per_unit` column, which is the primary source for unit price, is not always populated. When it is blank, there is no direct unit price available in the row.

**Fix:**  
`get_unit_price` tries three methods in order and uses the first one that produces a result:
1. Parse `iga_price_per_unit` directly (e.g. `'$1.47/100g'` → `'0.0147 per g'`)
2. Divide `iga_price_numeric` by `iga_unit_of_measure_size` (e.g. `$15.80 / 1 kg`)
3. Divide `iga_price_numeric` by `iga_unit_of_measure_type` as a last resort

If all three methods fail for a row, `unit_price` will be `NaN` for that record.

---

#### 4. Some IGA products have no match in the DiscountMate products table
**Issue:**  
Not every product scraped from IGA exists in our products table in MongoDB. This can happen if the product was never catalogued, or if the barcode in the scrape doesn't match any known barcode in our database.

**Fix**
`join_prods` performs an inner join so only records with a matching barcode in the products table are kept. Unmatched products are silently dropped.

**To investigate:**  
Whether unmatched products should be flagged and logged for the team to review, rather than silently dropped. This would help identify gaps in the products table over time.